# 05 · Feature engineering
**Step 7:** build business features from the data **read back out of SQL**:

| Level | Features |
|---|---|
| Order | total order value, delivery days, delivery delay, delayed flag, order month, primary category / payment type |
| Customer | order count, total spending, average order value, repeat-customer indicator, spending tier |
| Seller | revenue, order count, units sold, average item price, average review score |

Definitions are documented at the top of `src/features.py`.

In [ ]:
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src import config
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid", palette="viridis")

# Where the raw CSVs live. To rehearse on generated test data instead, use:
#   RAW_DIR = config.SYNTHETIC_DIR   (run `python scripts/make_synthetic_data.py` first)
RAW_DIR = config.RAW_DIR

In [ ]:
from src.db import get_engine
from sqlalchemy import text

engine = get_engine()
print("Connected:", engine.url.render_as_string(hide_password=True))


def query(sql, params=None):
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

In [ ]:
from src.load_to_sql import read_tables, load_features, write_meta
from src.features import build_all_features

tables = read_tables(engine)
features = build_all_features(tables)
for name, df in features.items():
    print(name, df.shape)

In [ ]:
of = features["order_features"]
display(of.head())
display(of[["total_order_value", "delivery_days", "delivery_delay_days"]].describe())

In [ ]:
display(features["customer_features"].describe(include="all").T.head(12))
display(features["seller_features"].sort_values("seller_revenue", ascending=False).head(10))

## Store the feature tables in SQL

In [ ]:
DATA_SOURCE = "real"      # use "synthetic" if you loaded generated test data
load_features(engine, features)
write_meta(engine, DATA_SOURCE)
display(query("SELECT COUNT(*) AS order_features FROM order_features"))